# Trying different off-the-shelf OCR tools

## 1. Deepseek OCR 2

Problem: Deepseek OCR 2 (and 1) requires CUDA.

## 2. GLM OCR

In [ ]:
# The current XPU runtime limits single allocations to ~4GB by default. The following line resolves this issue.
import os
os.environ['UR_L0_ENABLE_RELAXED_ALLOCATION_LIMITS'] = '1'

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from PIL import Image
torch.xpu.empty_cache()

image = Image.open("/media/jan/LWL_Kultur/A01_a-Adel/1/09_Ääbääre/00001145_1#A01_1_09_Ääbääre.jpg")


MODEL_PATH = "zai-org/GLM-OCR"
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image,#"url": "/media/jan/LWL_Kultur/A01_a-Adel/1/09_Ääbääre/00001145_1#A01_1_09_Ääbääre.jpg"
            },
            {
                "type": "text",
                "text": "Text Recognition:"
            }
        ],
    }
]

device = 'xpu' if torch.xpu.is_available() else 'cpu'
processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = AutoModelForImageTextToText.from_pretrained(
    pretrained_model_name_or_path=MODEL_PATH,
    torch_dtype=torch.float16,
    device_map=device,
)

print(model.device)

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

inputs.pop("token_type_ids", None)
generated_ids = model.generate(**inputs, max_new_tokens=128)
output_text = processor.decode(generated_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print(output_text)

Problem: Takes ages to run / doesn’t stop.

## 3. PaddleOCR

In [ ]:
from PIL import Image
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

# ---- Settings ----
model_path = "PaddlePaddle/PaddleOCR-VL-1.5"
image_path = "/media/jan/LWL_Kultur/A01_a-Adel/1/09_Ääbääre/00001145_1#A01_1_09_Ääbääre.jpg"
task = "ocr" # Options: 'ocr' | 'table' | 'chart' | 'formula' | 'spotting' | 'seal'
# ------------------

# ---- Image Preprocessing For Spotting ----
image = Image.open(image_path).convert("RGB")
image.thumbnail((768, 768))
orig_w, orig_h = image.size
spotting_upscale_threshold = 1500

if task == "spotting" and orig_w < spotting_upscale_threshold and orig_h < spotting_upscale_threshold:
    process_w, process_h = orig_w * 2, orig_h * 2
    try:
        resample_filter = Image.Resampling.LANCZOS
    except AttributeError:
        resample_filter = Image.LANCZOS
    image = image.resize((process_w, process_h), resample_filter)

# Set max_pixels: use 1605632 for spotting, otherwise use default ~1M pixels
max_pixels = 2048 * 28 * 28 if task == "spotting" else 768 * 28 * 28
# ---------------------------

# -------- Inference --------
#DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE = 'xpu' if torch.xpu.is_available() else 'cpu'
PROMPTS = {
    "ocr": "OCR:",
    "table": "Table Recognition:",
    "formula": "Formula Recognition:",
    "chart": "Chart Recognition:",
    "spotting": "Spotting:",
    "seal": "Seal Recognition:",
}

model = AutoModelForImageTextToText.from_pretrained(model_path, torch_dtype=torch.bfloat16, device_map='auto').eval()
processor = AutoProcessor.from_pretrained(model_path)

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": PROMPTS[task]},
        ]
    }
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    images_kwargs={"size": {"shortest_edge": processor.image_processor.min_pixels, "longest_edge": max_pixels}},
).to(DEVICE)

outputs = model.generate(**inputs, max_new_tokens=32)
result = processor.decode(outputs[0][inputs["input_ids"].shape[-1]:-1])
print(result)
# ---------------------------

Genau dasselbe Problem

## 4. Pure PaddleOCR pipeline

Endlich etwas funktionierendes, ist aber leider immer noch recht langsam.

In [11]:
from paddleocr import PaddleOCR

image_path = "/media/jan/LWL_Kultur/A01_a-Adel/1/09_Ääbääre/00001147_1#A01_1_09_Ääbääre.jpg"

# Initialize PaddleOCR instance
ocr = PaddleOCR(
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    enable_mkldnn=False)

# Run OCR inference on a sample image 
result = ocr.predict(
    input=image_path)

for res in result:
    text = ' '.join(res['rec_texts'])
    print(text)
    res.save_to_img("output")

Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/jan/.paddlex/official_models/PP-OCRv5_server_det`.
Creating model: ('PP-OCRv5_server_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `/home/jan/.paddlex/official_models/PP-OCRv5_server_rec`.


tbär 11043 ,Mnbar, Lougbin, Aonfe mt t it'  fn? Mn tri Roye vignt, Mnnn Sii pogoe gignt, Mnm sii yiln galw Bow tri dipir. voggl. E. H. W. Meyer Windheim 0.180
